In [8]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [2]:
!pip install solara

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


In [ ]:
!du -h

In [1]:
#Modified Cohort TBI_Having_Epilepsy_Cohort to include TBI_Date -> Original Cohort
Epilepsy_Cohort_new = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Having_Epilepsy_Cohort_Add_TBIDate')

In [3]:
Epilepsy_Cohort_new.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- EPI_date: string (nullable = true)



In [ ]:
Epilepsy_Cohort = Epilepsy_Cohort_new.toPandas()

In [ ]:
Epilepsy_Cohort.to_pickle("Epilepsy_Cohort.pkl")

In [2]:
!pip install solara

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


In [3]:
import solara

In [4]:
path = "Epilepsy_Cohort.pkl"
with open(path, 'rb') as f:
    data = f.read()

 
solara.FileDownload(data=data, label="Download file", filename="Epilepsy_Cohort.pkl")

Cannot show ipywidgets in text

In [ ]:
Epilepsy_Cohort_new.createOrReplaceTempView("Epilepsy_Cohort")

In [ ]:
#Original Control
Epilepsy_Control = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NotHaving_Epilepsy_Control')

In [ ]:
Epilepsy_Control.printSchema()

In [ ]:
Epilepsy_Control.createOrReplaceTempView("Epilepsy_Control")

In [ ]:
Epilepsy_Cohort_months = spark.sql("""SELECT
    personid,
    TBI_date,
    EPI_date,
    MONTHS_BETWEEN(EPI_date,TBI_date) AS mh_date
FROM
    Epilepsy_Cohort

""")

# Show the result
Epilepsy_Cohort_months.show(2, truncate=False)

In [ ]:
Epilepsy_Cohort_months = spark.sql("""SELECT
    personid,
    TBI_date,
    EPI_date,
    CASE
        WHEN TBI_date <= EPI_date THEN
            (YEAR(EPI_date) - YEAR(TBI_date)) * 12 + (MONTH(EPI_date) - MONTH(TBI_date))
        ELSE
            NULL -- or any other appropriate value for cases where TBI_date is greater than EPI_date
    END AS mh_date
FROM
    Epilepsy_Cohort
""")
# Show the result
Epilepsy_Cohort_months.show(2, truncate=False)

In [ ]:
Epilepsy_Cohort_months.createOrReplaceTempView("Epilepsy_Cohort_months")

In [ ]:
Epilepsy_Cohort_months_T = spark.sql("""SELECT * from Epilepsy_Cohort_months where mh_date = '' or mh_date = NULL""")
# Show the result
Epilepsy_Cohort_months_T.show(2, truncate=False)

In [ ]:
Epilepsy_Cohort_months.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_months")

In [ ]:
# # Modify-1
# Epilepsy_Control_Latest_DiagDate = spark.sql("""
# SELECT distinct
#     t.personid,
#     t.TBI_date,
#     r.conditioncode.standard.id as diagnosis_code,
#     r.effectivedate as diagnosis_date
# FROM
#     Epilepsy_Control t
# INNER JOIN
#     condition r
# ON
#     t.personid = r.personid
# WHERE
#     r.conditioncode.standard.id NOT IN ('G40%', '345%', 'R56.%')  
#     OR r.conditioncode.standard.id NOT IN ('780.39', 'R55', '780.2', 'Q04.3', '742.4')
# """)

# Show the result
Epilepsy_Control_Latest_DiagDate.show(5, truncate=False)

Epilepsy_Control_Latest_DiagDate = spark.sql("""
SELECT
    t.personid,
    t.TBI_date,
    MAX(CASE WHEN r.effectivedate = '' THEN NULL ELSE r.effectivedate END) AS latest_diagdate
FROM
    Epilepsy_Control t
INNER JOIN
    condition r
ON
    t.personid = r.personid
GROUP BY
    t.personid, t.TBI_date
""")

# Show the result
Epilepsy_Control_Latest_DiagDate.show(5, truncate=False)

In [ ]:
Epilepsy_Control_Latest_DiagDate.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Latest_DiagDate")

In [ ]:
#Tested against Not in EPI
Epilepsy_Control_Latest_DiagDate = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Latest_DiagDate')

In [ ]:
#Tested against Not in EPI
EPI_MED = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epi_med_Edited')
EPI_MED.createOrReplaceTempView("Epilepsy_Cohort_Med")

In [ ]:
Epilepsy_Control_Latest_DiagDate.createOrReplaceTempView("Epilepsy_Control_Latest")

In [ ]:
# Epilepsy_Control_Latest_DiagDate.printSchema()
EPI_MED.printSchema()

In [ ]:
#  Modify-1
Epilepsy_Control_Latest_DiagDate_R = spark.sql("""
SELECT distinct
    t.personid,
    t.TBI_date,
    t.diagnosis_code,
    t.diagnosis_date
FROM
    Epilepsy_Control_Latest t
LEFT JOIN
    Epilepsy_Cohort_Med r
ON
    t.personid = r.personid
WHERE
    r.personid IS NULL
""")

# Show the result
Epilepsy_Control_Latest_DiagDate_R.show(5, truncate=False)

In [ ]:
#Tested against Not in EPI
Epilepsy_Control_Latest_DiagDate_R.write.mode("overwrite").parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Latest_DiagMed')

In [3]:
#Tested against Not in EPI
Epilepsy_Control_Latest_DiagDate_R1 = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Latest_DiagMed')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
Epilepsy_Control = Epilepsy_Control_Latest_DiagDate_R1.toPandas()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o116.collectToPython.
: java.lang.OutOfMemoryError: GC overhead limit exceeded


In [ ]:
Epilepsy_Control.to_pickle("Epilepsy_Control.pkl")

In [ ]:
import solara
 
path = "Epilepsy_Control.pkl"
with open(path, 'rb') as f:
    data = f.read()

 
solara.FileDownload(data=data, label="Download file", filename="Epilepsy_Control.pkl")

In [ ]:
Epilepsy_Control_Latest_DiagDate_R2 = Epilepsy_Control_Latest_DiagDate_R1.drop('diagnosis_code')

In [ ]:
Epilepsy_Control_Latest_DiagDate_R2.printSchema()
Epilepsy_Control_Latest_DiagDate_R2.createOrReplaceTempView("Epilepsy_Control_meddiag")

In [ ]:
Epilepsy_Control_testing = spark.sql(""" 
SELECT distinct
    personid,
    TBI_date,
    MAX(diagnosis_date) AS diagnosis_date
FROM
    Epilepsy_Control_meddiag
GROUP BY
    personid, TBI_date

""")
# Show the result
Epilepsy_Control_testing.show(5, truncate=False)

In [ ]:
#Tested against Not in EPI
Epilepsy_Control_testing.write.mode("overwrite").parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_testingdata')

In [ ]:
#Tested against Not in EPIEpilepsy_Control_Latest_DiagDate_Result
Epilepsy_Control_Latest_DiagDate_Result = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_testingdata')

In [ ]:
Epilepsy_Control_Latest_DiagDate_Result.createOrReplaceTempView("Epilepsy_Control_compare")

In [ ]:
Epilepsy_Control_Latest_DiagDate_Result.printSchema()

In [ ]:
Epilepsy_Control_md = Epilepsy_Control.drop('TBI_date')
Epilepsy_Control_md.createOrReplaceTempView("Epilepsy_Control_md")
Epilepsy_Control_md.printSchema()

In [ ]:
Control_testdata_1 = spark.sql(""" 
SELECT COUNT(personid) AS record_count
FROM Epilepsy_Control_md
""")

Control_testdata_2 = spark.sql(""" 
SELECT COUNT(personid) AS record_count
FROM Epilepsy_Control_compare
""")
# Show the result
Control_testdata_1.show(truncate=False)
Control_testdata_2.show(truncate=False)

In [ ]:
Control_testdata_1 = spark.sql(""" 
SELECT *
FROM Epilepsy_Control_md
""")

Control_testdata_2 = spark.sql(""" 
SELECT COUNT(personid) AS record_count
FROM Epilepsy_Control_compare
""")
# Show the result
Control_testdata_1.show(truncate=False)
Control_testdata_2.show(truncate=False)

In [ ]:
# Modify-1
ControlTest = spark.sql("""
SELECT count(*) as count
FROM
    Epilepsy_Control t
INNER JOIN
    condition r
ON
    t.personid = r.personid
WHERE
    r.conditioncode.standard.id IN ('G40%', '345%', 'R56.%')  
    OR r.conditioncode.standard.id IN ('780.39', 'R55', '780.2', 'Q04.3', '742.4')
""")

# Show the result
ControlTest.show(truncate=False)

In [ ]:
ControlTest = spark.sql("""
SELECT count(*) as count
FROM
    Epilepsy_Control t
INNER JOIN
    Epilepsy_Cohort_Med r
ON
    t.personid = r.personid
""")

# Show the result
ControlTest.show(truncate=False)

In [ ]:
Epilepsy_Control_Latest_DiagDate_R1.createOrReplaceTempView("Epilepsy_Control_meddiag_2")
Epilepsy_Control_testing.createOrReplaceTempView("Epilepsy_Control_meddiag_1")
Epilepsy_Control_testing2 = spark.sql(""" 
SELECT distinct
    t.personid,
    t.TBI_date,
    t.diagnosis_date,
    r.diagnosis_code
FROM
    Epilepsy_Control_meddiag_1 t
INNER JOIN
    Epilepsy_Control_meddiag_2 r
on
    t.diagnosis_date = r.diagnosis_date
    AND
    t.personid = r.personid
""")
# Show the result
Epilepsy_Control_testing2.show(5, truncate=False)

In [ ]:
Epilepsy_Control_testing2.createOrReplaceTempView("Epilepsy_Control_meddiag_final")
Epilepsy_Control_testing3 = spark.sql(""" 
SELECT personid, COUNT(*) as count
FROM Epilepsy_Control_meddiag_final
GROUP BY personid
HAVING count > 1 
""")
# Show the result
Epilepsy_Control_testing3.show(truncate=False)

In [ ]:
Epilepsy_Control_testing3 = spark.sql(""" 
SELECT *
FROM Epilepsy_Control_meddiag_final
where personid = 'da26844b-3b2d-4d9d-9d37-f7a5128612e0'
""")
# Show the result
Epilepsy_Control_testing3.show(truncate=False)

In [ ]:
Epilepsy_Control_testing4 = spark.sql(""" 
SELECT distinct
    t.personid,
    t.TBI_date,
    r.diagnosis_date,
    t.diagnosis_code
FROM
    Epilepsy_Control_meddiag_final t
INNER JOIN
    Epilepsy_Control_meddiag_2 r
on
    t.diagnosis_code = r.diagnosis_code
    AND
    t.personid = r.personid
""")
# Show the result
Epilepsy_Control_testing4.show(5, truncate=False)

In [ ]:
Epilepsy_Control_testing4.createOrReplaceTempView("Epilepsy_Control_meddiag_final1")
Epilepsy_Control_testing5 = spark.sql(""" 
SELECT *
FROM Epilepsy_Control_meddiag_final1
where personid = 'da26844b-3b2d-4d9d-9d37-f7a5128612e0'
""")
# Show the result
Epilepsy_Control_testing5.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col, when, first

latest_diagnoses = spark.sql("""
SELECT DISTINCT
    personid,
    TBI_date,
    CASE WHEN diagnosis_code RLIKE '^[A-Za-z]' THEN diagnosis_code ELSE NULL END AS diagnosis_code,
    MIN(diagnosis_date) AS diagnosis_date
FROM
    Epilepsy_Control_meddiag_final1
GROUP BY
    personid, TBI_date, diagnosis_code
""")

latest_diagnoses.show(truncate=False)

In [ ]:
#exexute -1- No need to execute this cell anymore
latest_diagnoses_1 = spark.sql("""SELECT DISTINCT
    personid,
    TBI_date,
    CASE WHEN diagnosis_code RLIKE '^[A-Za-z]' THEN diagnosis_code ELSE NULL END AS diagnosis_code,
    MAX(diagnosis_date) AS diagnosis_date
FROM
    Epilepsy_Control_months
GROUP BY
    personid, TBI_date, diagnosis_code """)

latest_diagnoses_1.show(truncate=False)

In [ ]:
Epilepsy_Control_Latest_DiagDate.createOrReplaceTempView("Epilepsy_Control_months")
Epilepsy_Control_testing = spark.sql("""
    SELECT personid, count(*) FROM Epilepsy_Control_months GROUP BY personid HAVING count(*) > 1
""")
# Show the result
Epilepsy_Control_testing.show(5, truncate=False)


In [ ]:
# Modify-1
Epilepsy_Control_Latest_Diganosiscode = spark.sql("""
SELECT distinct
    t.personid,
    t.TBI_date,
    t.latest_diagdate,
    r.conditioncode.standard.id as latest_diagnosis_code
FROM
    Epilepsy_Control_months t
INNER JOIN
    condition r
ON
    t.personid = r.personid
    where t.latest_diagdate = r.effectivedate
""")

# Show the result
Epilepsy_Control_Latest_Diganosiscode.show(5, truncate=False)

In [ ]:
Epilepsy_Control_Latest_DiagDate.printSchema()
Epilepsy_Control_Latest_Diganosiscode.createOrReplaceTempView("Epilepsy_Control_diagcode")
Epilepsy_Control_Latest_Diganosiscode.printSchema()

In [ ]:
# Modify-1
Epilepsy_Control_Latest_Diganosiscode_1 = spark.sql("""
SELECT
    personid,
    TBI_date,
    latest_diagdate,
    conditioncode.standard.id as latest_diagnosis_code
FROM
    Epilepsy_Control_diagcode

""")

# Show the result
Epilepsy_Control_Latest_Diganosiscode_1.show(5, truncate=False)

In [ ]:
#Modified Cohort TBI_Having_Epilepsy_Cohort to include TBI_Date
Epilepsy_Cohort_new = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Having_Epilepsy_Cohort_Add_TBIDate')

In [ ]:
Epilepsy_Cohort = Epilepsy_Cohort_new.toPandas()

In [ ]:
#Modify-2

Epilepsy_Control_months = spark.sql("""SELECT
    personid,
    TBI_date,
    latest_diagdate,
    CASE
        WHEN TBI_date <= latest_diagdate THEN
            (YEAR(latest_diagdate) - YEAR(TBI_date)) * 12 + (MONTH(latest_diagdate) - MONTH(TBI_date))
        ELSE
            NULL -- or any other appropriate value for cases where TBI_date is greater than latest_diagdate
    END AS mh_date
FROM
    Epilepsy_Control_months
""")
# Show the result
Epilepsy_Control_months.show(5, truncate=False)

In [ ]:
#Modify-3
##extracted_date_df.createOrReplaceTempView('extracted_date_df')
control_demo =spark.sql("""
select distinct l.personid, l.birthdate, l.gender,l.race
from
dedupedemographics l
inner join
Epilepsy_Control_months r
on l.personid = r.personid""")

In [ ]:
##Modify-4
#Extracting Demo
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
# Extract the date using regexp_extract
extracted_date_df = control_demo.withColumn("gender", col("gender.value")).withColumn("birthdate", col("birthdate.value").cast(DateType())).withColumn("race", col("race.value"))

In [ ]:
##Modify-5
extracted_date_df.createOrReplaceTempView('extracted_date_df')
control_demo =spark.sql("""
select distinct l.personid, l.birthdate, l.gender,l.race
from
extracted_date_df l
inner join
Epilepsy_Control_months r
on l.personid = r.personid""")

In [ ]:
##Modify-6
#write -control_demo - Overridden with latest change to include TBI_Date and Latest DiagDate
control_demo.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_Stacked_mh")

In [ ]:
##Modify-7
#Edited for latest cohort to include age at TBI Diagnosis
control_demo_Unprocessed = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_Stacked_mh")

In [ ]:
##Modify-8
control_demo_Unprocessed.createOrReplaceTempView("demo_Unprocessed")

In [ ]:
##Modify-9
control_demo_Unprocessed.printSchema()

In [ ]:
#Modify-10
Epilepsy_Control_months.printSchema()

In [ ]:
#Execute again
def process_demo(spark, cohort, demo):
    demo.createOrReplaceTempView('demo_temp')
    cohort.createOrReplaceTempView('cohort_temp')    
    
    demo_processed = spark.sql("""
    select distinct l.personid, r.mh_date,
        CASE
            WHEN l.birthdate <> '' THEN l.birthdate
            ELSE NULL
        END AS birthdate,
        CASE
            WHEN l.gender = 'Male' THEN 'Male'
            WHEN l.gender = 'Female' THEN 'Female'
            WHEN l.gender = 'None' THEN Null
            ELSE 'other_gender'
        END AS gender,
        CASE
            WHEN l.race = 'White' THEN 'White'
            WHEN l.race in ('Black','African American') Then 'Black'
            WHEN l.race = 'Asian' Then 'Asian'
            WHEN l.race IN ('Hispanic, black', 'Hispanic, white', 'Hispanic') THEN 'Hispanic'
            WHEN l.race IN ('Native American',' American Indian','Alaska Native', 'Native Hawaiian',
            'Other Pacific Islander',
            'Central American Indian', 'Canadian and Latin American Indian', 'Spanish American Indian', 'South American Indian', 
            'Mexican American Indian', 'Alaskan Indian','Asian or Pacific islander','American Indian or Alaska Native',
            'Native Hawaiian   or Other Pacific Islander') THEN 'Native American'
            ELSE 'Other_race'
        END AS race,
        '1' AS conditioncode,
        r.latest_diagdate,
        r.TBI_date
    FROM demo_temp l
    inner join cohort_temp r
    on l.personid = r.personid
    order by l.personid
    """)
#     demo_processed = demo_processed.withColumn('age_at_EPI_diagnosis',lit(months_between(col('EPI_date'),col('birthdate'))/12))
    demo_processed = demo_processed.withColumn('age_of_TBI_diagnosis',lit(months_between(col('TBI_date'),col('birthdate'))/12))
    demo_processed.cache()      
    return demo_processed

In [ ]:
#Modify-11
from pyspark.sql.functions import *
demo_processed = process_demo(spark, Epilepsy_Control_months, control_demo_Unprocessed)

In [ ]:
#Modify-12
demo_processed.printSchema()

In [ ]:
#Modify-13
#Edited for adding age_at_TBI_diagDate
demo_processed.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Demo_Processed_mh_ld")

In [ ]:
control_demo_1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Demo_Processed_mh_ld")

In [ ]:
control_demo_1.createOrReplaceTempView("control_demo")

In [ ]:
control_demo_1.printSchema()

In [ ]:
Epilepsy_Control_R = spark.sql("""SELECT count(personid) from control_demo where age_of_TBI_diagnosis < 0

""")
# Show the result
Epilepsy_Control_R.show(truncate=False)

In [6]:
cohort_demo_1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_Demo_Processed_mh")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
cohort_demo_1.show(2, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------+----------+------+-----+-------------+-------------------------+-------------------------+--------------------+--------------------+
|personid                            |mh_date|birthdate |gender|race |conditioncode|EPI_date                 |TBI_date                 |age_at_EPI_diagnosis|age_of_TBI_diagnosis|
+------------------------------------+-------+----------+------+-----+-------------+-------------------------+-------------------------+--------------------+--------------------+
|304e20b3-36f9-44b4-9554-d22342f92a77|null   |2004-01-01|Male  |White|1            |2021-11-27T04:10:00+00:00|2022-02-06T19:00:00+00:00|17.903692503333335  |18.098902329999998  |
|304e8f2a-1ae2-4bf8-904f-56cd7cd31cb0|49     |1932-07-08|Male  |White|1            |2020-04-06T20:29:33+00:00|2016-03-04T08:00:00+00:00|87.7469189625       |83.65681003583333   |
+------------------------------------+-------+----------+------+-----+-------------+---------------------

<IPython.core.display.Javascript object>

In [ ]:
cohort_demo_1.createOrReplaceTempView("cohort_demo")

In [7]:
cohort_demo_1.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)



In [ ]:
Epilepsy_Cohort_cnt = spark.sql("""SELECT count(personid) from cohort_demo""")
# Show the result
Epilepsy_Cohort_cnt.show(truncate=False)

In [ ]:
Epilepsy_Control_cnt = spark.sql("""SELECT count(personid) from control_demo""")
# Show the result
Epilepsy_Control_cnt.show(truncate=False)

In [ ]:
Epilepsy_Cohort_R = spark.sql("""SELECT count(personid) from cohort_demo where age_of_TBI_diagnosis < 0
""")
# Show the result
Epilepsy_Cohort_R.show(truncate=False)

In [ ]:
Epilepsy_Cohort_R = spark.sql("""SELECT count(personid) from control_demo where age_of_TBI_diagnosis < 0
""")
# Show the result
Epilepsy_Cohort_R.show(truncate=False)

In [ ]:
Epilepsy_Cohort_R = spark.sql("""SELECT count(personid) from control_demo where mh_date < 0
""")
# Show the result
Epilepsy_Cohort_R.show(truncate=False)

In [ ]:
Epilepsy_Cohort_R1 = spark.sql("""
    SELECT *
    FROM cohort_demo
    WHERE age_of_TBI_diagnosis >= 0
""")

# Show the result
Epilepsy_Cohort_R1.show(truncate=False)
Epilepsy_Control_R = spark.sql("""
    SELECT *
    FROM control_demo
    WHERE age_of_TBI_diagnosis >= 0
""")

# Show the result
Epilepsy_Control_R.show(truncate=False)

In [ ]:
Epilepsy_Cohort_R1.createOrReplaceTempView("cohort_demo_1")
Epilepsy_Control_R.createOrReplaceTempView("control_demo_1")
Epilepsy_Cohort_R1.printSchema()
Epilepsy_Control_R.printSchema()

In [ ]:
#write -Cohort_demo - Overridden with latest Cohort with age at TBI Diagnosis
Epilepsy_Cohort_R1.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_Stacked_mh_dna")

In [ ]:
#write -Cohort_demo - Overridden with latest Cohort with age at TBI Diagnosis
Epilepsy_Control_R.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_Stacked_mh_dna")

In [2]:
#write -Cohort_demo - Overridden with latest Cohort with age at TBI Diagnosis
Epilepsy_Cohort_M = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_Stacked_mh_dna")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
Epilepsy_Cohort_M.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)



In [4]:
Epilepsy_Cohort_M.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+----------+------+---------------+-------------+-------------------------+-------------------------+--------------------+--------------------+
|personid                            |mh_date|birthdate |gender|race           |conditioncode|EPI_date                 |TBI_date                 |age_at_EPI_diagnosis|age_of_TBI_diagnosis|
+------------------------------------+-------+----------+------+---------------+-------------+-------------------------+-------------------------+--------------------+--------------------+
|304e20b3-36f9-44b4-9554-d22342f92a77|null   |2004-01-01|Male  |White          |1            |2021-11-27T04:10:00+00:00|2022-02-06T19:00:00+00:00|17.903692503333335  |18.098902329999998  |
|304e8f2a-1ae2-4bf8-904f-56cd7cd31cb0|49     |1932-07-08|Male  |White          |1            |2020-04-06T20:29:33+00:00|2016-03-04T08:00:00+00:00|87.7469189625       |83.65681003583333   |
|304f3d53-c8e4-4bb7-a2f6-ff80fa19c16b|1      |1939-04-1

<IPython.core.display.Javascript object>

In [5]:
print(Epilepsy_Cohort_M.distinct().count())
print(Epilepsy_Cohort_M.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

190415


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

190415


<IPython.core.display.Javascript object>

In [ ]:
#write -Control_demo - Overridden with latest Cohort with age at TBI Diagnosis
Epilepsy_Control_M = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_Stacked_mh_dna")

In [ ]:
Epilepsy_Cohort_M.createOrReplaceTempView("cohort_demo_2")
Epilepsy_Control_M.createOrReplaceTempView("control_demo_2")
Epilepsy_Cohort_M.printSchema()
Epilepsy_Control_M.printSchema()

In [ ]:
Epilepsy_Cohort_R1 = spark.sql("""SELECT count(personid) from cohort_demo_1 where age_of_TBI_diagnosis < 0
""")
# Show the result
Epilepsy_Cohort_R1.show(truncate=False)

In [ ]:
Epilepsy_Control_R1 = spark.sql("""SELECT count(personid) from control_demo_1 where age_of_TBI_diagnosis < 0
""")
# Show the result
Epilepsy_Control_R1.show(truncate=False)

In [ ]:
Epilepsy_Cohort_cnt_1 = spark.sql("""SELECT count(personid) from cohort_demo_1""")
# Show the result
Epilepsy_Cohort_cnt_1.show(truncate=False)

In [ ]:
Epilepsy_Control_cnt_1 = spark.sql("""SELECT count(personid) from control_demo_1""")
# Show the result
Epilepsy_Control_cnt_1.show(truncate=False)

In [ ]:
#Final Paired cohort:
Epilepsy_Cohort_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT distinct
        race,
        gender,
        age_of_TBI_diagnosis,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned,
        mh_date
    FROM cohort_demo_2
    WHERE age_of_TBI_diagnosis >= 0 -- Remove negative values
),
control_cte AS (
    SELECT distinct
        race,
        gender,
        age_of_TBI_diagnosis,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned_control, 
        mh_date
    FROM control_demo_2
    WHERE age_of_TBI_diagnosis >= 0 -- Remove negative values
)
SELECT distinct
    cohort_cte.age_of_TBI_diagnosis,
    cohort_cte.race,
    cohort_cte.gender,
    cohort_cte.mh_date
FROM cohort_cte
JOIN control_cte ON
    cohort_cte.age_binned = control_cte.age_binned_control
    AND cohort_cte.mh_date = control_cte.mh_date
    AND cohort_cte.race = control_cte.race
    AND cohort_cte.gender = control_cte.gender
""")
# Show the result
Epilepsy_Cohort_Pairing_R.show(5,truncate=False)

In [ ]:
#Final Paired control:
Epilepsy_Control_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT distinct
        race,
        gender,
        age_of_TBI_diagnosis,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned,
        mh_date
    FROM cohort_demo_2
    WHERE age_of_TBI_diagnosis >= 0 -- Remove negative values
),
control_cte AS (
    SELECT distinct
        race,
        gender,
        age_of_TBI_diagnosis,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned_control, 
        mh_date
    FROM control_demo_2
    WHERE age_of_TBI_diagnosis >= 0 -- Remove negative values
)
SELECT distinct
    control_cte.age_of_TBI_diagnosis,
    control_cte.race,
    control_cte.gender,
    control_cte.mh_date
FROM control_cte
JOIN cohort_cte ON
    cohort_cte.age_binned = control_cte.age_binned_control
    AND cohort_cte.mh_date = control_cte.mh_date
    AND cohort_cte.race = control_cte.race
    AND cohort_cte.gender = control_cte.gender
""")
# Show the result
Epilepsy_Control_Pairing_R.show(5,truncate=False)

In [ ]:
Epilepsy_Control_Pairing_R.createOrReplaceTempView("control_demo_paired")

In [ ]:
#Final control pair and original control_demo_2 joined for all columns
Epilepsy_Pairing_RRC = spark.sql("""
   SELECT t.personid, t.birthdate, t.conditioncode, t.TBI_date, t.latest_diagdate, t.age_of_TBI_diagnosis, t.race, t.gender, t.mh_date FROM control_demo_2 t 
   INNER JOIN control_demo_paired r ON 
       t.age_of_TBI_diagnosis = r.age_of_TBI_diagnosis 
       AND t.race = r.race 
       AND t.gender = r.gender 
       AND t.mh_date = r.mh_date
""")
# Show the result
Epilepsy_Pairing_RRC.show(truncate=False)

In [ ]:
#write -Cohort_demo - Overridden with latest Control with age at TBI Diagnosis
Epilepsy_Pairing_RRC.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_paired_Final")

In [ ]:
Epilepsy_Cohort_Pairing_R.printSchema()

In [ ]:
Epilepsy_Cohort_Pairing_R.createOrReplaceTempView("cohort_demo_paired")

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT Count(*) FROM cohort_demo_paired 
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT * FROM cohort_demo_paired 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis < 20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT Count(*) FROM cohort_demo_paired 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis < 20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT Count(*) FROM cohort_demo_2 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis < 20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT Count(*) FROM control_demo_2 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis <20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT age_of_TBI_diagnosis, race, gender, mh_date FROM cohort_demo_2 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis <20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
#Final cohort pair and original cohort_demo_2 joined for all columns
Epilepsy_Pairing_RR = spark.sql("""
   SELECT t.personid, t.birthdate, t.conditioncode, t.EPI_date, t.TBI_date, t.age_of_TBI_diagnosis, t.age_at_EPI_diagnosis, t.race, t.gender, t.mh_date FROM cohort_demo_2 t 
   INNER JOIN cohort_demo_paired r ON 
       t.age_of_TBI_diagnosis = r.age_of_TBI_diagnosis 
       AND t.race = r.race 
       AND t.gender = r.gender 
       AND t.mh_date = r.mh_date
""")
# Show the result
Epilepsy_Pairing_RR.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R.createOrReplaceTempView("cohort_demo_paired_All")

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT age_of_TBI_diagnosis, race, gender, mh_date FROM cohort_demo_paired_All 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis <20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT Count(*) FROM cohort_demo_paired_All 

""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
#write -Cohort_demo - Overridden with latest Cohort with age at TBI Diagnosis
Epilepsy_Pairing_RR.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_paired_Final")

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT age_of_TBI_diagnosis, race, gender, mh_date FROM cohort_demo_paired 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis <20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
   SELECT age_of_TBI_diagnosis, race, gender, mh_date FROM control_demo_2 
   WHERE race = 'Asian' 
   AND gender = 'Female' 
   AND mh_date = 0 
   AND age_of_TBI_diagnosis >= 15 
   AND age_of_TBI_diagnosis <20
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_control_binned
    FROM control_demo
    GROUP BY age_binned_control, mh_control_binned, race, gender
)
   SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo_1
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any mhistory above 100 (optional)
    END AS mh_control_binned
    FROM control_demo_1
    GROUP BY age_binned_control, mh_control_binned, race, gender
)
   SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender where cohort_cte.cohort_cnt > control_cte.control_cnt
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo_1
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any mhistory above 100 (optional)
    END AS mh_control_binned
    FROM control_demo_1
    GROUP BY age_binned_control, mh_control_binned, race, gender
),
   tempview_1 as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select count(*) from tempview_1
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo_1
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any mhistory above 100 (optional)
    END AS mh_control_binned
    FROM control_demo_1
    GROUP BY age_binned_control, mh_control_binned, race, gender
),
   tempview as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender where cohort_cte.cohort_cnt > control_cte.control_cnt)
        select count(*) from tempview
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo_1
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any mhistory above 100 (optional)
    END AS mh_control_binned
    FROM control_demo_1
    GROUP BY age_binned_control, mh_control_binned, race, gender
),
   tempview_1 as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select sum(cohort_cnt), sum(control_cnt) from tempview_1
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS mh_binned
    FROM cohort_demo_1
    GROUP BY age_binned, mh_binned, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control,
    CASE
        WHEN mh_date < 5 THEN '0-4'
        WHEN mh_date >= 5 AND mh_date < 10 THEN '5-9'
        WHEN mh_date >= 10 AND mh_date < 15 THEN '10-14'
        WHEN mh_date >= 15 AND mh_date < 20 THEN '15-19'
        WHEN mh_date >= 20 AND mh_date < 25 THEN '20-24'
        WHEN mh_date >= 25 AND mh_date < 30 THEN '25-29'
        WHEN mh_date >= 30 AND mh_date < 35 THEN '30-34'
        WHEN mh_date >= 35 AND mh_date < 40 THEN '35-39'
        WHEN mh_date >= 40 AND mh_date < 45 THEN '40-44'
        WHEN mh_date >= 45 AND mh_date < 50 THEN '45-49'
        WHEN mh_date >= 50 AND mh_date < 55 THEN '50-54'
        WHEN mh_date >= 55 AND mh_date < 60 THEN '55-59'
        WHEN mh_date >= 60 AND mh_date < 65 THEN '60-64'
        WHEN mh_date >= 65 AND mh_date < 70 THEN '65-69'
        WHEN mh_date >= 70 AND mh_date < 75 THEN '70-74'
        WHEN mh_date >= 75 AND mh_date < 80 THEN '75-79'
        WHEN mh_date >= 80 AND mh_date < 85 THEN '80-84'
        WHEN mh_date >= 85 AND mh_date < 90 THEN '85-89'
        WHEN mh_date >= 90 AND mh_date < 95 THEN '90-94'
        WHEN mh_date >= 95 AND mh_date <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any mhistory above 100 (optional)
    END AS mh_control_binned
    FROM control_demo_1
    GROUP BY age_binned_control, mh_control_binned, race, gender
),
   tempview_1 as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_binned,
        control_cte.mh_control_binned
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_binned = control_cte.mh_control_binned
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select sum(cohort_cnt), sum(control_cnt) from tempview_1 where cohort_cnt > control_cnt
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    mh_date
    FROM cohort_demo_2
    GROUP BY age_binned, mh_date, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control, mh_date
    FROM control_demo_2
    GROUP BY age_binned_control, mh_date, race, gender
),
   tempview as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_date,
        control_cte.mh_date
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_date = control_cte.mh_date
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender where cohort_cte.cohort_cnt > control_cte.control_cnt)
        select count(*) from tempview
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    mh_date
    FROM cohort_demo_2
    GROUP BY age_binned, mh_date, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control, 
    mh_date
    FROM control_demo_2
    GROUP BY age_binned_control, mh_date, race, gender
),
   tempview as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_date,
        control_cte.mh_date as mh_control
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_date = control_cte.mh_date
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select count(*) from tempview
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
#Testing for pair up count
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT
        count(*) as cohort_cnt,
        race,
        gender,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned,
        mh_date
    FROM cohort_demo_2
    GROUP BY age_binned, mh_date, race, gender
), control_cte AS (
    SELECT
        count(*) as control_cnt,
        race,
        gender,
        CASE
            WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
            WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
            WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
            WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
            WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
            WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
            WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
            WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
            WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
            WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
            WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
            WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
            WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
            WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
            WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
            WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
            WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
            WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
            WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
            WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
            ELSE 'over_100' -- Handle any age above 100 (optional)
        END AS age_binned_control, 
        mh_date
    FROM control_demo_2
    GROUP BY age_binned_control, mh_date, race, gender
)
SELECT  
    cohort_cte.cohort_cnt,
    control_cte.control_cnt,
    cohort_cte.race,
    control_cte.race AS race_control,
    cohort_cte.gender,
    control_cte.gender AS gender_control,
    cohort_cte.age_binned,
    control_cte.age_binned_control,
    cohort_cte.mh_date,
    control_cte.mh_date as mh_control
FROM cohort_cte
JOIN control_cte ON 
    cohort_cte.age_binned = control_cte.age_binned_control
    AND cohort_cte.mh_date = control_cte.mh_date
    AND cohort_cte.race = control_cte.race
    AND cohort_cte.gender = control_cte.gender
WHERE cohort_cte.age_binned = '15-19'
AND cohort_cte.race = 'Asian'
AND cohort_cte.gender = 'Female'
AND cohort_cte.mh_date = '0'
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    mh_date
    FROM cohort_demo_2
    GROUP BY age_binned, mh_date, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control, mh_date
    FROM control_demo_2
    GROUP BY age_binned_control, mh_date, race, gender
),
   tempview as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_date,
        control_cte.mh_date
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_date = control_cte.mh_date
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select sum(cohort_cnt), sum(control_cnt) from tempview
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_cte AS (
    SELECT count(*) as cohort_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned,
    mh_date
    FROM cohort_demo_1
    GROUP BY age_binned, mh_date, race, gender
), control_cte AS (
    SELECT count(*) as control_cnt, race, gender, CASE
        WHEN age_of_TBI_diagnosis < 5 THEN '0-4'
        WHEN age_of_TBI_diagnosis >= 5 AND age_of_TBI_diagnosis < 10 THEN '5-9'
        WHEN age_of_TBI_diagnosis >= 10 AND age_of_TBI_diagnosis < 15 THEN '10-14'
        WHEN age_of_TBI_diagnosis >= 15 AND age_of_TBI_diagnosis < 20 THEN '15-19'
        WHEN age_of_TBI_diagnosis >= 20 AND age_of_TBI_diagnosis < 25 THEN '20-24'
        WHEN age_of_TBI_diagnosis >= 25 AND age_of_TBI_diagnosis < 30 THEN '25-29'
        WHEN age_of_TBI_diagnosis >= 30 AND age_of_TBI_diagnosis < 35 THEN '30-34'
        WHEN age_of_TBI_diagnosis >= 35 AND age_of_TBI_diagnosis < 40 THEN '35-39'
        WHEN age_of_TBI_diagnosis >= 40 AND age_of_TBI_diagnosis < 45 THEN '40-44'
        WHEN age_of_TBI_diagnosis >= 45 AND age_of_TBI_diagnosis < 50 THEN '45-49'
        WHEN age_of_TBI_diagnosis >= 50 AND age_of_TBI_diagnosis < 55 THEN '50-54'
        WHEN age_of_TBI_diagnosis >= 55 AND age_of_TBI_diagnosis < 60 THEN '55-59'
        WHEN age_of_TBI_diagnosis >= 60 AND age_of_TBI_diagnosis < 65 THEN '60-64'
        WHEN age_of_TBI_diagnosis >= 65 AND age_of_TBI_diagnosis < 70 THEN '65-69'
        WHEN age_of_TBI_diagnosis >= 70 AND age_of_TBI_diagnosis < 75 THEN '70-74'
        WHEN age_of_TBI_diagnosis >= 75 AND age_of_TBI_diagnosis < 80 THEN '75-79'
        WHEN age_of_TBI_diagnosis >= 80 AND age_of_TBI_diagnosis < 85 THEN '80-84'
        WHEN age_of_TBI_diagnosis >= 85 AND age_of_TBI_diagnosis < 90 THEN '85-89'
        WHEN age_of_TBI_diagnosis >= 90 AND age_of_TBI_diagnosis < 95 THEN '90-94'
        WHEN age_of_TBI_diagnosis >= 95 AND age_of_TBI_diagnosis <= 100 THEN '95-100'
        ELSE 'over_100' -- Handle any age above 100 (optional)
    END AS age_binned_control, mh_date
    FROM control_demo_1
    GROUP BY age_binned_control, mh_date, race, gender
),
   tempview as (SELECT 
        cohort_cte.cohort_cnt,
        control_cte.control_cnt,
        cohort_cte.race,
        control_cte.race AS race_control,
        cohort_cte.gender,
        control_cte.gender AS gender_control,
        cohort_cte.age_binned,
        control_cte.age_binned_control,
        cohort_cte.mh_date,
        control_cte.mh_date
    FROM cohort_cte
    JOIN control_cte ON 
        cohort_cte.age_binned = control_cte.age_binned_control
        AND cohort_cte.mh_date = control_cte.mh_date
        AND cohort_cte.race = control_cte.race
        AND cohort_cte.gender = control_cte.gender)
        select sum(cohort_cnt), sum(control_cnt) from tempview where cohort_cnt > control_cnt
""")
# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
WITH cohort_counts AS (
    SELECT
        COUNT(*) as cohort_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN
        control_demo_1 AS t
    ON
        c.gender = t.gender
        AND c.race = t.race       
        AND c.age_of_TBI_diagnosis = t.age_of_TBI_diagnosis
),

control_counts AS (
    SELECT
        COUNT(*) as control_pid_cnt
    FROM
        control_demo_1 AS t
    INNER JOIN
        cohort_demo_1 AS c
    ON
        c.gender = t.gender
        AND c.race = t.race        
        AND c.age_of_TBI_diagnosis = t.age_of_TBI_diagnosis
)

SELECT
    cohort_counts.cohort_pid_cnt,
    control_counts.control_pid_cnt
FROM
    cohort_counts
CROSS JOIN
    control_counts
""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
    SELECT distinct
        c.gender,
        c.race,
        cohort_count.cohort_pid_cnt,
        control_count.control_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN (
        SELECT
            gender,
            race,
            COUNT(*) as cohort_pid_cnt
        FROM
            cohort_demo_1
        GROUP BY
            gender, race
    ) AS cohort_count
    ON
        c.gender = cohort_count.gender
        AND c.race = cohort_count.race
    INNER JOIN (
        SELECT
            gender,
            race,
            COUNT(*) as control_pid_cnt
        FROM
            control_demo_1
        GROUP BY
            gender, race
    ) AS control_count
    ON
        c.gender = control_count.gender
        AND c.race = control_count.race
""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
    SELECT
        c.gender,
        c.race,
        c.mh_date,
        COUNT(*) as cohort_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN
        control_demo_1 AS t
    ON
        c.gender = t.gender
        AND c.race = t.race  
        AND c.mh_date = t.mh_date
    GROUP BY
        c.gender, c.race, c.mh_date
""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R2 = spark.sql("""
    SELECT
        COUNT(*) as cohort_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN
        control_demo_1 AS t
    ON
        c.gender = t.gender
        AND c.race = t.race  
        AND c.age_of_TBI_diagnosis = t.age_of_TBI_diagnosis
        AND c.mh_date = t.mh_date

""")

# Show the result
Epilepsy_Pairing_R2.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
    SELECT
        COUNT(*) as cohort_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN
        control_demo_1 AS t
    ON
        c.gender = t.gender
        AND c.race = t.race  
        AND c.age_of_TBI_diagnosis = t.age_of_TBI_diagnosis

""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
    SELECT distinct
        c.gender,
        c.race,
        c.age_of_TBI_diagnosis,
        c.mh_date,    
        cohort_count.cohort_pid_cnt,
        control_count.control_pid_cnt
    FROM
        cohort_demo AS c
    INNER JOIN (
        SELECT
            gender,
            race,
            age_of_TBI_diagnosis,
            mh_date,
            COUNT(*) as cohort_pid_cnt
        FROM
            cohort_demo
        GROUP BY
            gender, race, age_of_TBI_diagnosis, mh_date
    ) AS cohort_count
    ON
        c.gender = cohort_count.gender
        AND c.race = cohort_count.race
        AND c.age_of_TBI_diagnosis = cohort_count.age_of_TBI_diagnosis
        AND c.mh_date = cohort_count.mh_date
    INNER JOIN (
        SELECT
            gender,
            race,
            age_of_TBI_diagnosis,
            mh_date,
            COUNT(*) as control_pid_cnt
        FROM
            control_demo
        GROUP BY
            gender, race, age_of_TBI_diagnosis, mh_date
    ) AS control_count
    ON
        c.gender = control_count.gender
        AND c.race = control_count.race
        AND c.age_of_TBI_diagnosis = control_count.age_of_TBI_diagnosis
        AND c.mh_date = control_count.mh_date
        
""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [ ]:
Epilepsy_Pairing_R = spark.sql("""
    SELECT
        COUNT(*) as cohort_pid_cnt
    FROM
        cohort_demo_1 AS c
    INNER JOIN
        control_demo_1 AS t
    ON
        c.gender = t.gender
        AND c.race = t.race  
        AND c.mh_date = t.mh_date
""")

# Show the result
Epilepsy_Pairing_R.show(truncate=False)

In [6]:
Epilepsy_Cohort_Paired_New = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_paired_Final")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
Epilepsy_Control_Paired_New = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_paired_Final_ld")

In [2]:
Epilepsy_Cohort_Paired_New.printSchema()
# Epilepsy_Control_Paired_New.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)



In [ ]:
Epilepsy_Cohort_Paired_New.createOrReplaceTempView('Epilepsy_Cohort_Paired')
Epilepsy_Control_Paired_New.createOrReplaceTempView('Epilepsy_Control_Paired')

In [ ]:
Epilepsy_Cohort_Demo_Result1 = spark.sql("SELECT count(personid) FROM Epilepsy_Cohort_Paired where age_of_TBI_diagnosis < 18")
Epilepsy_Cohort_Demo_Result1.show(truncate = False)
Epilepsy_Cohort_Demo_Result1 = spark.sql("SELECT count(personid) FROM Epilepsy_Cohort_Paired where age_of_TBI_diagnosis >= 18 and age_of_TBI_diagnosis < 44")
Epilepsy_Cohort_Demo_Result1.show(truncate = False)
Epilepsy_Cohort_Demo_Result1 = spark.sql("SELECT count(personid) FROM Epilepsy_Cohort_Paired where age_of_TBI_diagnosis >= 44 and age_of_TBI_diagnosis < 60")
Epilepsy_Cohort_Demo_Result1.show(truncate = False)
Epilepsy_Cohort_Demo_Result1 = spark.sql("SELECT count(personid) FROM Epilepsy_Cohort_Paired where age_of_TBI_diagnosis >= 60")
Epilepsy_Cohort_Demo_Result1.show(truncate = False)

In [1]:
# Epilepsy_Cohort_Demo_R.createOrReplaceTempView('Cohort_Demo_r')
# Epilepsy_Cohort_Demo_R.printSchema()
# Epilepsy_Cohort_Demo_Result1 = spark.sql("SELECT * FROM Cohort_Demo_r where age_of_TBI_diagnosis < 0")
# Epilepsy_Cohort_Demo_Result1.show(truncate = False)
#Over all cohort and control comparison before pairing:
Epilepsy_Control_Demo_R = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_Demo_Edited_1")
Epilepsy_Cohort_Demo_R = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_Demo_Processed_edited_1")

In [2]:
Epilepsy_Cohort_Demo_R.printSchema()
Epilepsy_Control_Demo_R.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- diagnosis_date: string (nullable = true)
 |-- age_at_diagnosis: double (nullable = true)



In [9]:
#Finalized DS to use for after pairing
import numpy as np
from pyspark.sql.functions import col
from tabulate import tabulate

def cohort_distribution_Final_float(spark, cohort_demo):
    cohort_demo = cohort_demo.distinct()
    cohort_demo.cache()

    # Calculate gender counts and percentages
    genderCnt = cohort_demo.select('personid', 'gender').na.drop().distinct().groupBy('gender').count()
    total_gender_count = genderCnt.agg({'count': 'sum'}).collect()[0][0]
    genderCnt = genderCnt.withColumn('percentage', (col('count') / total_gender_count) * 100)

    # Calculate race counts and percentages
    raceCnt = cohort_demo.select('personid', 'race').na.drop().distinct().groupBy('race').count()
    total_race_count = raceCnt.agg({'count': 'sum'}).collect()[0][0]
    raceCnt = raceCnt.withColumn('percentage', (col('count') / total_race_count) * 100)

    # Calculate the total count of distinct individuals before age filtering
    total_individuals_count = cohort_demo.count()
    print('total_individuals_count', total_individuals_count)

    age_ranges = ['age<18', '18<=age<44', '44<=age<60', 'age>=60']  # Age ranges as strings
    age_statistics = []  # To store mean and standard deviation for each age range

    # Calculate counts, percentages, mean, and standard deviation for each age range
    for age_range in age_ranges:
        if age_range == 'age<18':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis < 18)
        elif age_range == '18<=age<44':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 18) & (cohort_demo.age_of_TBI_diagnosis < 44))
        elif age_range == '44<=age<60':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 44) & (cohort_demo.age_of_TBI_diagnosis < 60))
        elif age_range == 'age>=60':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis >= 60)

        age_count = age_data.count()

        # Calculate percentage using the total count before age filtering
        age_percentage = (age_count / total_individuals_count) * 100

        # Calculate sample statistics
        age_data = age_data.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
        sample_mean_age = np.mean(age_data)
        sample_std_dev_age = np.std(age_data, ddof=1)

        age_statistics.append([age_range, age_count, f"{age_percentage:.2f}%", f"{sample_mean_age:.2f}", f"{sample_std_dev_age:.2f}"])

    # Calculate mean and standard deviation across all age ranges
    all_age_data = cohort_demo.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
    all_age_mean = np.mean(all_age_data)
    all_age_std_dev = np.std(all_age_data, ddof=1)

    age_statistics.append(['All Ages', total_individuals_count, '100.00%', f"{all_age_mean:.2f}", f"{all_age_std_dev:.2f}"])

    # Print gender table
    print("Gender Table:")
    print(tabulate(genderCnt.toPandas(), headers=['Gender', 'Count', 'Percentage']))

    # Print race table
    print("Race Table:")
    print(tabulate(raceCnt.toPandas(), headers=['Race', 'Count', 'Percentage']))

    # Print age table
    print("Age Table:")
    print(tabulate(age_statistics, headers=['Age Range', 'Count', 'Percentage', 'Mean Age', 'Standard Deviation'], tablefmt="grid"))

# Example usage
# cohort_demo = spark.read.csv("path/to/your/dataset.csv", header=True, inferSchema=True)
# cohort_distribution_Final_float(cohort_demo)

▸,:,


In [ ]:
#Finalized DS to use for before pairing
print(Epilepsy_Cohort_Demo_R.count())
print(Epilepsy_Control_Demo_R.count())

In [8]:
cohort_distribution_Final_float(spark, Epilepsy_Cohort_Paired_New)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    Race               Count    Percentage
--  ---------------  -------  ------------
 0  Native American     2047       1.34318
 1  Other_race         31664      20.7769
 2  White             109105      71.5912
 3  Hispanic            5626       3.6916
 4  Black               2009       1.31824
 5  Asian               1949       1.27887
Age Table:
+-------------+---------+--------------+------------+----------------------+
| Age Range   |   Count | Percentage   |   Mean Age |   Standard Deviation |
+=============+=========+==============+============+======================+
| age<18      |   40072 | 26.29%       |       7.98 |                 5.91 |
+-------------+---------+--------------+------------+----------------------+
| 18<=age<44  |   41417 | 27.18%       |      30.45 |                 7.49 |
+-------------+---------+--------------+------------+----------------------+
| 44<=age<60  |   26362 | 17.30%       |      52.36 |                 4.54 |
+-------------+---------+-------

<IPython.core.display.Javascript object>

In [10]:
Cohort_StratSample = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSample")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
Control_StratSample = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StratifiedSample2")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
Cohort_StratSample.printSchema()

root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)


In [7]:
Control_StratSample.printSchema()

root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Z21: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- X04: long (nullable = true)
 |-- F25: lo

In [12]:
# cohort_distribution_Final_float(spark, Epilepsy_Control_Paired_New)
cohort_distribution_Final_float(spark, Cohort_StratSample)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    Race               Count    Percentage
--  ---------------  -------  ------------
 0  Native American      198       1.29598
 1  Other_race          3164      20.7095
 2  White              10983      71.8877
 3  Hispanic             523       3.42322
 4  Black                214       1.40071
 5  Asian                196       1.28289
Age Table:
+-------------+---------+--------------+------------+----------------------+
| Age Range   |   Count | Percentage   |   Mean Age |   Standard Deviation |
+=============+=========+==============+============+======================+
| age<18      |    4023 | 26.33%       |       7.97 |                 5.9  |
+-------------+---------+--------------+------------+----------------------+
| 18<=age<44  |    4157 | 27.21%       |      30.4  |                 7.46 |
+-------------+---------+--------------+------------+----------------------+
| 44<=age<60  |    2668 | 17.46%       |      52.22 |                 4.54 |
+-------------+---------+------

<IPython.core.display.Javascript object>

In [16]:
# cohort_distribution_before_Final_float(spark, Epilepsy_Cohort_Demo_R)
cohort_distribution_Final_float(spark, Control_StratSample)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    Race               Count    Percentage
--  ---------------  -------  ------------
 0  Native American      772      0.765919
 1  Other_race         26419     26.2109
 2  White              67342     66.8115
 3  Hispanic            3953      3.92186
 4  Black                791      0.784769
 5  Asian               1517      1.50505
Age Table:
+-------------+---------+--------------+------------+----------------------+
| Age Range   |   Count | Percentage   |   Mean Age |   Standard Deviation |
+=============+=========+==============+============+======================+
| age<18      |   49666 | 49.27%       |       8.2  |                 5.78 |
+-------------+---------+--------------+------------+----------------------+
| 18<=age<44  |   26935 | 26.72%       |      28.27 |                 7.47 |
+-------------+---------+--------------+------------+----------------------+
| 44<=age<60  |    9104 | 9.03%        |      51.98 |                 4.6  |
+-------------+---------+----------

<IPython.core.display.Javascript object>

In [ ]:
cohort_distribution_before_Final_float(spark, Epilepsy_Control_Demo_R)

In [ ]:
# Control_testdata_1 = spark.sql(""" 
# SELECT * 
# FROM Epilepsy_Control_compare
# """)

Control_testdata_2 = spark.sql(""" 
SELECT *
FROM Epilepsy_Control_compare where personid in ('00143779-cb7c-4cf8-9354-6e130e79cee2', '001daaf0-a1a3-44e2-b8eb-d84cc4639d73', '00612a99-c11d-4798-8fd5-df6338336526', '0064575b-c2c4-4294-a33c-798ddc51f556', '006c9353-4e80-47ad-b895-c61e6c25ce8c')
""")
# Show the result
# Control_testdata_1.show(5, truncate=False)
Control_testdata_2.show(truncate=False)

In [ ]:
Control_testdata_2 = spark.sql(""" 
SELECT *
FROM control_demo where personid in ('00143779-cb7c-4cf8-9354-6e130e79cee2', '001daaf0-a1a3-44e2-b8eb-d84cc4639d73', '00612a99-c11d-4798-8fd5-df6338336526', '0064575b-c2c4-4294-a33c-798ddc51f556', '006c9353-4e80-47ad-b895-c61e6c25ce8c')
""")
Control_testdata_2.show(truncate=False)

In [ ]:
Control_testdata_2 = spark.sql(""" 
SELECT *
FROM Epilepsy_Control_Paired where personid in ('00143779-cb7c-4cf8-9354-6e130e79cee2', '001daaf0-a1a3-44e2-b8eb-d84cc4639d73', '00612a99-c11d-4798-8fd5-df6338336526', '0064575b-c2c4-4294-a33c-798ddc51f556', '006c9353-4e80-47ad-b895-c61e6c25ce8c')
""")
Control_testdata_2.show(truncate=False)
